# Tune Hyperparameters

Tunes hyperparameters using optuna and saves them as "best_hp_reg.json"

In [ ]:
import numpy as np
import pandas as pd
import json, os, random, pickle
from copy import deepcopy
from types import SimpleNamespace

import torch
import torch.nn.functional as F
from torch.nn import Linear, Dropout, BatchNorm1d, Sequential, ReLU
from torch_geometric.nn import GCNConv, GINEConv, GATv2Conv, global_mean_pool
from torch_geometric.loader import DataLoader
from torch.optim import Adam

from scipy.special import expit
from scipy.optimize import minimize_scalar

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    average_precision_score, precision_score,
    recall_score, classification_report, roc_curve
)
from xgboost import XGBClassifier

import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parents[0]))

import pbsg
from pbsg.polymer import polymers_from_df
from pbsg.polymer.generators import smiles_from_graphs
from pbsg.ml import to_pyg
from pbsg.ml.pyg import architecture_onehot
from pbsg.polymer.parser import parse_smiles_list
import ast

def safe_parse(val):
    try:
        if pd.isna(val): return None
    except (TypeError, ValueError): pass
    while isinstance(val, str):
        try: val = ast.literal_eval(val)
        except (ValueError, SyntaxError): break
    return val

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

In [ ]:
df = pd.read_csv('data_regressor.csv')
for col in ['beads_list', 'lengths', 'reactivity_ratios', 'spatiality_rules']:
    if col in df.columns:
        df[col] = df[col].apply(safe_parse)
df['lengths']             = df['lengths'].apply(lambda x: None if isinstance(x, str) else x)
df['architecture_onehot'] = df.apply(architecture_onehot, axis=1)
df['beads_list']          = df['repeat_units'].apply(parse_smiles_list)

In [ ]:
with open("random_stratified_splits_reg.json") as f:
    splits = json.load(f)

folds        = splits["folds"]
test_idx     = splits["test"]
all_idx      = np.arange(len(df))
trainval_idx = np.setdiff1d(all_idx, test_idx)
print(f"trainval={len(trainval_idx)}  test={len(test_idx)}  folds={len(folds)}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
X_fp_pooled_RU    = np.load("Vectors/X_fp_pooled_RU.npy")
X_fp_pooled_poly = np.load("Vectors/X_fp_pooled_poly.npy")
X_fp_RU = np.load("Vectors/X_fp_RU.npy")
X_fp_poly= np.load("Vectors/X_fp_poly.npy")

In [ ]:
graphs_PBSG = torch.load("Graphs/graphs_PBSG.pt")
graphs_SMILES=torch.load("Graphs/graphs_SMILES.pt")
graphs_SMILES_glob=torch.load("Graphs/graphs_SMILES+global.pt")

In [ ]:
y=np.load("Vectors/y.npy")

In [ ]:
for i, g in enumerate(graphs_PBSG):
    g.y = torch.tensor([y[i]], dtype=torch.float32)

for i, g in enumerate(graphs_SMILES):
    g.y = torch.tensor([y[i]], dtype=torch.float32)

for i, g in enumerate(graphs_SMILES_glob):
    g.y = torch.tensor([y[i]], dtype=torch.float32)

In [ ]:
from scripts.optuna_tuning_full_reg import tune_all_reg

tune_all_reg(
    graphs_PBSG      = graphs_PBSG,
    graphs_SMILES    = graphs_SMILES,
    graphs_SMILES_gl = graphs_SMILES_glob,
    X_fp_pooled_RU   = X_fp_pooled_RU,
    X_fp_pooled_poly = X_fp_pooled_poly,
    X_fp_RU          = X_fp_RU,
    X_fp_poly        = X_fp_poly,
    y=y, folds=folds, df=df, device=device,
    n_trials=30,
)